# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Divyanshu10045/flyrank-assignments/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

DRAFT

---

The frame is ML-04's, unchanged. One row is one (client, content) pair at one decision point,
**2026-03-01**. Features come from 2026-01-01 → 2026-02-28 (59 days). The label is March, and March
is not allowed anywhere near the score — I check that in Section 4 rather than promise it.
Section 0 is mine: two signal checks, before the rule, so the rule is frozen from measured
verdicts rather than from taste.

I'm building the rule a human can read out loud, because the whole point of this week is a baseline
that ML-08 has to beat. If my rule can't be said in three sentences, I haven't understood it yet.

**One thing I have to flag before anything else.** I went looking for the staleness signal and it
isn't available. The obvious one is `days_since_last_update`, and it's the first condition in
FlyRank's own `stale_visible_page` reason code:

```python
scripts/02_baseline_score.py:25-26
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
```

I can't use it, and the reason isn't a guess. `dim_content` is a current snapshot, so 78% of my
rows carry a `content_updated_date` *after* the decision point (ML-04 measured it). "Days since
update" would be reading the future and calling it freshness. So there is no staleness signal in
this notebook, and Section 1 is where I say what I used instead.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Setup: same local-run discipline as ML-04, because this box refuses duckdb.
import importlib
import os
import time
from pathlib import Path

REQUIRED = {"pyarrow": "pyarrow", "huggingface_hub": "huggingface_hub",
            "python-dotenv": "dotenv", "pandas": "pandas", "numpy": "numpy"}
_missing = []
for _pip, _mod in REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ImportError:
        _missing.append(_pip)
if _missing:
    raise SystemExit("missing packages: %s -> pip install %s" % (_missing, " ".join(_missing)))

import dotenv
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "work" / "notebooks").is_dir() and (p / ".git").exists()), None)
if REPO is None:
    raise SystemExit("could not find the repo root from %s" % Path.cwd())
os.chdir(REPO)
print("repo root:", REPO)
print("pandas %s | numpy %s | pyarrow %s" % (pd.__version__, np.__version__, pa.__version__))

# The read token comes from .env at the repo root, which .gitignore already covers.
# It is read by name, never printed, never written into this cell.
dotenv.load_dotenv(REPO / ".env")
_HF = os.environ.get("HF_TOKEN", "")
if not _HF:
    raise SystemExit("HF_TOKEN not found. Put it in %s as HF_TOKEN=... and rerun." % (REPO / ".env"))
fs = HfFileSystem(token=_HF)
print("HF_TOKEN loaded: %s (length %d, value never printed)" % (bool(_HF), len(_HF)))

WAREHOUSE = "datasets/FlyRank/internship-warehouse"
FACT = WAREHOUSE + "/fact_content_daily_performance"
KEYS = ["client_hash_id", "content_hash_id"]

# Windows, copied from ML-04 rather than retyped from memory.
FEATURE_START, FEATURE_END = "2026-01-01", "2026-02-28"
LABEL_START, LABEL_END = "2026-03-01", "2026-03-31"
DECISION_POINT = LABEL_START
FEATURE_MONTHS = ("2026-01", "2026-02")
LABEL_MONTHS = ("2026-03",)
DAYS_IN = {"2026-01": 31, "2026-02": 28, "2026-03": 31}
PRIOR_DAYS = 59
LABEL_DAYS = 31

POOL_FLOOR, VOLUME_FLOOR, DROP_FLOOR = 200, 500, -20

# Dev-loop cache. Deliberately OUTSIDE the repo: a dataset file must never be
# written into work/ where it could ever be staged. If this exists, the monthly
# aggregates are reused; otherwise they are read from the warehouse and written here.
CACHE = Path(os.environ.get("TEMP", ".")) / "opencode" / "ml04_cache"
CACHE.mkdir(parents=True, exist_ok=True)
print("dev cache (outside the repo):", CACHE)

DAILY_COLS = ["report_date", "client_hash_id", "content_hash_id",
              "gsc_impressions", "gsc_clicks", "gsc_sum_position", "gsc_data_available"]


def month_file(m):
    parts = sorted(p for p in fs.ls(FACT + "/month=" + m, detail=False) if p.endswith(".parquet"))
    if not parts:
        raise FileNotFoundError("no parquet under month=%s" % m)
    return parts[0]


def agg_month(m, my_clients):
    # Per (client, content) sums over one month, or ML-04's cached copy of the same thing.
    # Same three filters ML-04 used, same reasons: my clients only, gsc_data_available
    # IS TRUE (three-valued flag, so IS TRUE is the only honest test), and
    # gsc_impressions > 0 (a day with no impressions means the page did not show up).
    #
    # ORDER MATTERS and I got it wrong the first time. The cache holds the WHOLE panel,
    # so the client filter has to run on the cached rows too - returning the cache
    # unfiltered silently widens the frame from 29 clients to 42 and changes the base
    # rate. The client filter is below, after the cache branch, on purpose.
    cached = CACHE / ("agg_%s.parquet" % m)
    if cached.exists():
        g = pd.read_parquet(cached)
        return g[g["client_hash_id"].isin(my_clients)].copy()
    t = pq.read_table(fs.open(month_file(m)), columns=DAILY_COLS)
    if my_clients:
        t = t.filter(pc.is_in(t.column("client_hash_id"),
                              value_set=pa.array(sorted(my_clients))))
    t = t.filter(pc.fill_null(t.column("gsc_data_available"), False))
    t = t.filter(pc.greater(t.column("gsc_impressions"), 0))
    g = t.group_by(KEYS).aggregate([("report_date", "count"),
                                    ("gsc_impressions", "sum"),
                                    ("gsc_clicks", "sum"),
                                    ("gsc_sum_position", "sum")]).to_pandas()
    g = g.rename(columns={"report_date_count": "active_days",
                          "gsc_impressions_sum": "impr",
                          "gsc_clicks_sum": "clicks",
                          "gsc_sum_position_sum": "sum_pos"})
    g["month"] = m
    g.to_parquet(cached)
    return g


print("\nready. windows: features %s -> %s (%d d), label %s -> %s (%d d)"
      % (FEATURE_START, FEATURE_END, PRIOR_DAYS, LABEL_START, LABEL_END, LABEL_DAYS))


C:\Users\dk732\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


repo root: C:\Users\dk732\Documents\FlyRank Intern\flyrank-assignments
pandas 3.0.5 | numpy 2.5.2 | pyarrow 25.0.1
HF_TOKEN loaded: True (length 37, value never printed)
dev cache (outside the repo): C:\Users\dk732\AppData\Local\Temp\opencode\ml04_cache

ready. windows: features 2026-01-01 -> 2026-02-28 (59 d), label 2026-03-01 -> 2026-03-31 (31 d)


## 0. Two signal checks (before the rule)

*Two things my rule leans on. Each one gets a bucket table, an outcome, and a verdict.*

DRAFT

I need to be clear about what these tables are and are not. Both signals are computed from
January and February only, so both are knowable at the decision point. The *outcome* column is
March — I'm allowed to look at March here, because this is the audit, not the rule. The rule in
Section 1 is frozen from these verdicts, and Section 5 scores it once.

Two things I could not do, so I'm not pretending to:

- **No staleness signal.** As flagged at the top, `days_since_last_update` is contaminated. The
  closest honest substitute is `content_age_days`, and I'm using that instead, out loud.
- **No FlyRank product flag as a feature.** `health_score`, `priority_score`, `action_type` are
  not in the data — the guide says so at `docs/ml-intern-dataset-and-lane-guide.md:186` — and it
  also says never to rebuild one and use it as an ordinary feature. So I'm borrowing the
  *conditions* from the starter's reason codes as things worth testing, and nothing more.

**Signal A — is the page under-clicking for the position it holds?**
Behind `low_ctr_visible_page`, `scripts/02_baseline_score.py:33-34`. This is the CTR-fix idea:
position in one side, CTR in the other.

**Signal B — is it already sliding inside the feature window?**
Behind `declining_with_demand`, `scripts/02_baseline_score.py:27-28`. FlyRank's version keys off
`trend_direction`, which in the starter CSV is a bucket inside the same 90 days as the features —
which is why ML-03 threw it out. I'm rebuilding the idea honestly: the slope from January's daily
impression rate to February's, both of which sit before the decision point.

The dictionary gives me the thresholds to copy rather than invent (`docs/data-dictionary.md:99`):
`down` is better than −20%, `up` is more than +20%.

**What I got, said up front so the cells below aren't a surprise.** Verdict A: **CONFIRMED**, but
weak and confounded with position — I'll show why I don't trust it enough to score on it. Verdict B:
**MIXED**. It looks strong against ML-04's label, and the label audit cell right after it explains
why that strength is mostly an artefact of how the label is built. ML-04's label measures March
against the **January+February blend**, so a page that already collapsed in February is
mechanically close to being a positive before March is even looked at. Signal B is not forecasting
March; it is largely re-measuring February. Mixed, not falsified: it does separate cleanly on the
label, and it does almost nothing on a forward March test.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the frame, reusing ML-04's cached monthly aggregates where they exist.
t0 = time.time()
clients = pq.read_table(fs.open(WAREHOUSE + "/dim_clients.parquet")).to_pandas()
gsc_start = pd.to_datetime(clients["gsc_data_start"].astype("string"))
MY_CLIENTS = set(clients.loc[(gsc_start <= FEATURE_START) & (clients["has_gsc_access"] != False),  # noqa: E712
                             "client_hash_id"])
print("clients kept (gsc covers the feature window, has_gsc_access IS NOT FALSE): %d" % len(MY_CLIENTS))

monthly = {m: agg_month(m, MY_CLIENTS) for m in FEATURE_MONTHS + LABEL_MONTHS}
for m, g in monthly.items():
    print("  month=%s -> %s page-clients (%s)"
          % (m, f"{len(g):,}", "from cache" if (CACHE / ('agg_%s.parquet' % m)).exists() else "read now"))

# ---- the two feature-month rates, and the slope between them -----------------
# Force the frame to the exact ML-04 client set to avoid any cache/widening drift.
# That means: if someone rebuilds the cache, we still match ML-04.
j = monthly[FEATURE_MONTHS[0]].rename(columns={"impr": "impr_jan", "clicks": "clicks_jan",
                                                "sum_pos": "sum_pos_jan", "active_days": "days_jan"})
b = monthly[FEATURE_MONTHS[1]].rename(columns={"impr": "impr_feb", "clicks": "clicks_feb",
                                                "sum_pos": "sum_pos_feb", "active_days": "days_feb"})
lb = monthly[LABEL_MONTHS[0]][KEYS + ["impr"]].rename(columns={"impr": "impr_label"})

_clients_after = {m: monthly[m]["client_hash_id"].nunique() for m in monthly}
print("clients present in each monthly aggregate after filtering: %s" % _clients_after)
print("client set size I asked for: %d" % len(MY_CLIENTS))
print("all monthly keys non-empty: %s" % all(len(monthly[m]) > 0 for m in monthly))

f = j.merge(b, on=KEYS, how="outer").merge(lb, on=KEYS, how="inner")
print("\npage-clients: jan|feb union %s, intersected with March %s"
      % (f"{len(f):,}", f"{len(f):,}"))

# Per-day, not per-month: January is 31 days and February is 28, so a raw sum comparison
# would read a stable page as a 10% drop for arithmetic reasons alone.
f["jan_rate_30d"] = f["impr_jan"].fillna(0) / DAYS_IN[FEATURE_MONTHS[0]] * 30
f["feb_rate_30d"] = f["impr_feb"].fillna(0) / DAYS_IN[FEATURE_MONTHS[1]] * 30
f["impr_prior"] = f["impr_jan"].fillna(0) + f["impr_feb"].fillna(0)
f["clicks_prior"] = f["clicks_jan"].fillna(0) + f["clicks_feb"].fillna(0)
f["sum_pos_prior"] = f["sum_pos_jan"].fillna(0) + f["sum_pos_feb"].fillna(0)
f["active_days_prior"] = f["days_jan"].fillna(0) + f["days_feb"].fillna(0)

# NAIVE WARNING: a raw Jan-vs-Feb sum ratio is the obvious version of this and it is
# wrong. Jan has 31 days, Feb has 28, so an unchanged page looks like -9.7% on sums.
_flat_page = (f["impr_jan"] > 0) & (f["impr_jan"] == f["impr_feb"])
print("sanity: %d pages have identical Jan and Feb impression SUMS. On those, a raw"
      % int(_flat_page.sum()))
print("        sum-vs-sum slope reads 0.0% by construction, while the per-day rate")
print("        change is %.1f%% - pure calendar arithmetic, 28 days against 31."
      % (DAYS_IN[FEATURE_MONTHS[1]] / DAYS_IN[FEATURE_MONTHS[0]] * 100 - 100))
print("        So I use per-day rates. A raw sum comparison would invent a ~10%")
print("        decline on every perfectly stable page in the panel.")

f["trend_pct_win"] = np.where(f["jan_rate_30d"] > 0,
                              (f["feb_rate_30d"] - f["jan_rate_30d"]) / f["jan_rate_30d"] * 100,
                              np.nan)

# ---- the three rates the label needs, all from the same two months ------------
f["prior_rate_30d"] = f["impr_prior"] / PRIOR_DAYS * 30
f["label_rate_30d"] = f["impr_label"] / LABEL_DAYS * 30
f["drop_pct"] = np.where(f["prior_rate_30d"] > 0,
                         (f["label_rate_30d"] - f["prior_rate_30d"]) / f["prior_rate_30d"] * 100,
                         np.nan)
f["lost_impr_30d"] = f["prior_rate_30d"] - f["label_rate_30d"]
f["ctr_prior_pct"] = np.where(f["impr_prior"] > 0, f["clicks_prior"] / f["impr_prior"] * 100, np.nan)
f["wtd_pos_prior"] = np.where(f["impr_prior"] > 0, f["sum_pos_prior"] / f["impr_prior"], np.nan)

content = pq.read_table(fs.open(WAREHOUSE + "/dim_content.parquet"),
                        columns=["content_hash_id", "content_created_date",
                                 "content_updated_date", "word_count"]).to_pandas()
for c in ("content_created_date", "content_updated_date"):
    content[c] = pd.to_datetime(content[c].astype("string"))
f = f.merge(content, on="content_hash_id", how="left")
f["content_age_days"] = (pd.Timestamp(DECISION_POINT) - f["content_created_date"]).dt.days

f["in_pool"] = f["impr_prior"] >= POOL_FLOOR
f["is_material_decline"] = (f["in_pool"] & (f["prior_rate_30d"] >= VOLUME_FLOOR)
                            & (f["drop_pct"] <= DROP_FLOOR)).astype(int)

print("\n=== frame reproduced (compare against ML-04) ===")
print("rows %s | clients %d | pool %s | material decline %s = %.4f"
      % (f"{len(f):,}", f["client_hash_id"].nunique(), f"{int(f['in_pool'].sum()):,}",
         f"{int(f.loc[f['in_pool'], 'is_material_decline'].sum()):,}",
         f.loc[f["in_pool"], "is_material_decline"].mean()))
print("ML-04 reported: rows 119,123 | pool 67,279 | positives 7,445 | base rate 0.1107")
print("               29 clients in the frame, 24 in the pool")
# Asserted, not eyeballed. My first run silently produced 137,900 rows across 42 clients
# because the cache holds the whole panel and I returned it before the client filter.
_ML04 = {"rows": 119123, "pool": 67279, "positives": 7445, "clients": 29}
_got = {"rows": len(f), "pool": int(f["in_pool"].sum()),
        "positives": int(f.loc[f["in_pool"], "is_material_decline"].sum()),
        "clients": f["client_hash_id"].nunique()}
for _k, _want in _ML04.items():
    print("  %-10s got %-8d ML-04 %-8d %s" % (_k, _got[_k], _want, "MATCH" if _got[_k] == _want else "MISMATCH"))
assert all(_got[_k] == _v for _k, _v in _ML04.items()), "frame drifted from ML-04"

P = f[f["in_pool"]].copy()
print("\n(%.1fs)" % (time.time() - t0))


clients kept (gsc covers the feature window, has_gsc_access IS NOT FALSE): 37


  month=2026-01 -> 115,641 page-clients (from cache)
  month=2026-02 -> 127,991 page-clients (from cache)
  month=2026-03 -> 145,190 page-clients (from cache)
clients present in each monthly aggregate after filtering: {'2026-01': 33, '2026-02': 32, '2026-03': 29}
client set size I asked for: 37
all monthly keys non-empty: True



page-clients: jan|feb union 119,123, intersected with March 119,123
sanity: 2034 pages have identical Jan and Feb impression SUMS. On those, a raw
        sum-vs-sum slope reads 0.0% by construction, while the per-day rate
        change is -9.7% - pure calendar arithmetic, 28 days against 31.
        So I use per-day rates. A raw sum comparison would invent a ~10%
        decline on every perfectly stable page in the panel.



=== frame reproduced (compare against ML-04) ===
rows 119,123 | clients 29 | pool 67,279 | material decline 7,445 = 0.1107
ML-04 reported: rows 119,123 | pool 67,279 | positives 7,445 | base rate 0.1107
               29 clients in the frame, 24 in the pool
  rows       got 119123   ML-04 119123   MATCH
  pool       got 67279    ML-04 67279    MATCH
  positives  got 7445     ML-04 7445     MATCH
  clients    got 29       ML-04 29       MATCH

(26.3s)


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- SIGNAL A: position x CTR -------------------------------------------------
print("=" * 78)
print("SIGNAL A - is it under-clicking for the position it holds?")
print("  flag borrowed: scripts/02_baseline_score.py:33-34 (low_ctr_visible_page)")
print("  'impressions_90d >= 500 and 0 < avg_position <= 20 and ctr < 0.5'")
print("=" * 78)
print("pool %s rows | base rate %.4f | ctr NaN %d | position NaN %d"
      % (f"{len(P):,}", P["is_material_decline"].mean(),
         int(P["ctr_prior_pct"].isna().sum()), int(P["wtd_pos_prior"].isna().sum())))
print("CTR here is a percentage, so the flag's ctr < 0.5 means under 0.5%.")
print()

P["pos_b"] = pd.cut(P["wtd_pos_prior"], [0, 5, 10, 20, np.inf],
                    labels=["pos<=5", "pos 5-10", "pos 10-20", "pos>20"])
P["ctr_b"] = pd.cut(P["ctr_prior_pct"], [-np.inf, 0.3, 0.5, np.inf],
                    labels=["ctr<0.3%", "ctr 0.3-0.5%", "ctr>=0.5%"])

POS_ORDER = ["pos<=5", "pos 5-10", "pos 10-20", "pos>20"]
CTR_ORDER = ["ctr<0.3%", "ctr 0.3-0.5%", "ctr>=0.5%"]
print("share of is_material_decline per cell, n in brackets:")
print("  %-10s %-16s %-16s %-16s" % ("", *CTR_ORDER))
for pb in POS_ORDER:
    row = "  %-10s" % pb
    for cb in CTR_ORDER:
        sub = P[(P["pos_b"] == pb) & (P["ctr_b"] == cb)]
        row += "  %-16s" % ("n/a" if len(sub) == 0 else "%.3f [%d]" % (sub["is_material_decline"].mean(), len(sub)))
    print(row)

print()
print("the two margins, with n printed, because a rate without n is decoration:")
for keys, name in ((["pos_b"], "by position band"), (["ctr_b"], "by ctr band")):
    g = P.groupby(keys, observed=True).agg(
        n=("is_material_decline", "size"), pos=("is_material_decline", "sum"),
        med_drop=("drop_pct", "median"), med_impr=("impr_prior", "median"))
    g["share"] = g["pos"] / g["n"]
    print("\n%s" % name)
    print(g.round(4).to_string())

print()
print("the flag's own cell, as literally written (pos<=20 and ctr<0.5):")
on = (P["wtd_pos_prior"] <= 20) & (P["ctr_prior_pct"] < 0.5)
off = (P["wtd_pos_prior"] <= 20) & (P["ctr_prior_pct"] >= 0.5)
for lab_, m in (("flag ON ", on), ("flag OFF", off)):
    print("  %s n=%-6d share=%.4f  median drop_pct=%+.1f%%"
          % (lab_, int(m.sum()), P.loc[m, "is_material_decline"].mean(), P.loc[m, "drop_pct"].median()))


SIGNAL A - is it under-clicking for the position it holds?
  flag borrowed: scripts/02_baseline_score.py:33-34 (low_ctr_visible_page)
  'impressions_90d >= 500 and 0 < avg_position <= 20 and ctr < 0.5'
pool 67,279 rows | base rate 0.1107 | ctr NaN 0 | position NaN 0
CTR here is a percentage, so the flag's ctr < 0.5 means under 0.5%.

share of is_material_decline per cell, n in brackets:
             ctr<0.3%         ctr 0.3-0.5%     ctr>=0.5%       
  pos<=5      0.169 [14549]     0.134 [4154]      0.130 [3925]    
  pos 5-10    0.137 [15385]     0.110 [3444]      0.101 [3360]    
  pos 10-20   0.067 [9733]      0.044 [1949]      0.057 [1903]    
  pos>20      0.030 [7410]      0.025 [961]       0.022 [506]     

the two margins, with n printed, because a rate without n is decoration:



by position band
               n   pos  med_drop  med_impr   share
pos_b                                             
pos<=5     22628  3520   16.3688    2364.5  0.1556
pos 5-10   22189  2827   19.3077    1565.0  0.1274
pos 10-20  13585   844   39.9750     959.0  0.0621
pos>20      8877   254   61.0911     650.0  0.0286

by ctr band
                  n   pos  med_drop  med_impr   share
ctr_b                                                
ctr<0.3%      47077  5430   23.9751    1291.0  0.1153
ctr 0.3-0.5%  10508  1045   31.4642    1825.0  0.0994
ctr>=0.5%      9694   970   32.1523    1975.5  0.1001

the flag's own cell, as literally written (pos<=20 and ctr<0.5):
  flag ON  n=49196  share=0.1267  median drop_pct=+21.3%
  flag OFF n=9206   share=0.1043  median drop_pct=+31.0%


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- SIGNAL B: the within-window slope ---------------------------------------
print("=" * 78)
print("SIGNAL B - is it already sliding, before the decision point?")
print("  flag borrowed: scripts/02_baseline_score.py:27-28 (declining_with_demand)")
print("  'trend_direction == down and impressions_90d >= 100'")
print("  mine is rebuilt honestly: per-day rate in Jan vs Feb, both inside the window")
print("=" * 78)


def trend_bucket(v):
    if pd.isna(v):
        return "1 no January baseline"
    if v < -20:
        return "2 down >20%"
    if v < 20:
        return "3 flat -20..+20"
    return "4 up >20%"


P["trend_b"] = P["trend_pct_win"].apply(trend_bucket)
g = P.groupby("trend_b").agg(n=("is_material_decline", "size"),
                             pos=("is_material_decline", "sum"),
                             med_drop=("drop_pct", "median"),
                             med_impr=("impr_prior", "median"),
                             med_jan=("jan_rate_30d", "median"),
                             med_feb=("feb_rate_30d", "median"))
g["share"] = g["pos"] / g["n"]
print("share of is_material_decline, and median drop_pct, per trend bucket:")
print(g.round(4).to_string())
print("\nbase rate for the pool: %.4f" % P["is_material_decline"].mean())

print()
print("n by bucket, so the floor is visible: " + ", ".join(
    "%s n=%d" % (b_, int(n)) for b_, n in g["n"].items()))
print("no bucket is near the 50-row floor; the smallest is %d." % int(g["n"].min()))

# The opposite check the skill asks for: a real signal survives on another slice.
print()
print("does the slope hold on a different slice? the top half of the pool by prior volume:")
top = P[P["impr_prior"] >= P["impr_prior"].median()]
g2 = top.groupby("trend_b").agg(n=("is_material_decline", "size"),
                                pos=("is_material_decline", "sum"))
g2["share"] = g2["pos"] / g2["n"]
print(g2.round(4).to_string())
print("base rate for that slice: %.4f" % top["is_material_decline"].mean())

print()
print("READ THIS BEFORE BELIEVING THE VERDICT: the next cell breaks this one open.")
print("The slope is CONFIRMED against the ML-04 label. I do not think that is a real")
print("forecasting signal, and the reason is in the label, not in the feature.")


SIGNAL B - is it already sliding, before the decision point?
  flag borrowed: scripts/02_baseline_score.py:27-28 (declining_with_demand)
  'trend_direction == down and impressions_90d >= 100'
  mine is rebuilt honestly: per-day rate in Jan vs Feb, both inside the window


share of is_material_decline, and median drop_pct, per trend bucket:
                           n   pos  med_drop  med_impr   med_jan    med_feb   share
trend_b                                                                            
1 no January baseline   4224    32  276.3046     669.5    0.0000   717.3214  0.0076
2 down >20%             9442  2504  -14.8822    1254.0  804.1935   440.3571  0.2652
3 flat -20..+20        15450  2345    5.2581    1876.5  951.7742   956.7857  0.1518
4 up >20%              38163  2564   43.7486    1479.0  453.8710  1042.5000  0.0672

base rate for the pool: 0.1107

n by bucket, so the floor is visible: 1 no January baseline n=4224, 2 down >20% n=9442, 3 flat -20..+20 n=15450, 4 up >20% n=38163
no bucket is near the 50-row floor; the smallest is 4224.

does the slope hold on a different slice? the top half of the pool by prior volume:
                           n   pos   share
trend_b                                   
1 no January baseline   1084    21

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- THE LABEL AUDIT. This is the most important cell in the notebook. --------
print("=" * 78)
print("LABEL AUDIT - is the slope forecasting March, or is it arithmetic?")
print("=" * 78)
print("The ML-04 label compares March to the Jan+Feb BLEND (prior_rate_30d).")
print("So a page that collapsed IN FEBRUARY keeps an inflated blend, and March then")
print("reads as a catastrophe even if March simply held February's level.")
print("If that is what is happening, the slope is not predicting anything - it is")
print("just measuring the same February collapse the label is partly built from.")
print()

# Reattach the two months so I can look at them separately.
K2 = KEYS
J = monthly[FEATURE_MONTHS[0]][K2 + ["impr"]].rename(columns={"impr": "ij"})
F_ = monthly[FEATURE_MONTHS[1]][K2 + ["impr"]].rename(columns={"impr": "ifeb"})
M_ = monthly[LABEL_MONTHS[0]][K2 + ["impr"]].rename(columns={"impr": "imar"})
# P already carries the rule's booleans, score, reason_code and action, so the alive
# slice below is ranked by the same score the queue uses.
Q = P.merge(J, on=K2, how="left").merge(F_, on=K2, how="left").merge(M_, on=K2, how="left")
print("Q carries the rule columns: %s"
      % [c for c in ("score", "is_sliding", "top_ten", "reason_code", "action") if c in Q.columns])
Q[["ij", "ifeb", "imar"]] = Q[["ij", "ifeb", "imar"]].fillna(0.0)
Q["r_jan"] = Q["ij"] / DAYS_IN[FEATURE_MONTHS[0]] * 30
Q["r_feb"] = Q["ifeb"] / DAYS_IN[FEATURE_MONTHS[1]] * 30
Q["r_mar"] = Q["imar"] / DAYS_IN[LABEL_MONTHS[0]] * 30

# A forward label that CANNOT see January. Not a replacement for the ML-04 label -
# a probe. It answers one question: does the Jan->Feb slope predict what March did?
Q["fwd_drop_pct"] = np.where(Q["r_feb"] > 0,
                             (Q["r_mar"] - Q["r_feb"]) / Q["r_feb"] * 100, np.nan)
Q["forward_decline"] = ((Q["r_feb"] >= VOLUME_FLOOR)
                        & (Q["fwd_drop_pct"] <= DROP_FLOOR)).astype(int)

# The alive slice: pages with real traffic in BOTH feature months. Without this the
# top of any trend ranking is just pages that went to zero in February, which is a
# different thing entirely and not a forecasting question.
alive = (Q["r_jan"] >= 250) & (Q["r_feb"] >= 250)
A = Q[alive].copy()
print("pages with >=250/mo in BOTH January and February: %s of %s (%.0f%%)"
      % (f"{int(alive.sum()):,}", f"{len(Q):,}", 100 * alive.mean()))
print()

t50 = A.nsmallest(50, "trend_pct_win")
print("the 50 steepest declines ON THAT SLICE, month by month:")
print("  median January rate %8.0f" % t50["r_jan"].median())
print("  median February rate %8.0f   <- already collapsed before anyone looked" % t50["r_feb"].median())
print("  median March rate    %8.0f   <- what actually happened next" % t50["r_mar"].median())
print()
print("  what the LABEL sees (March vs the Jan+Feb blend):")
print("    median drop_pct %+.1f%%" % t50["drop_pct"].median())
print("  what a HUMAN would call a further decline (March vs February):")
_f = (t50["r_mar"] - t50["r_feb"]) / t50["r_feb"] * 100
print("    median move     %+.1f%%" % _f.median())
print("    March below February in %d of 50; within +/-20%% of February in %d of 50"
      % (int((t50["r_mar"] < t50["r_feb"]).sum()), int((_f.abs() <= 20).sum())))
print()
print("=> February already fell off a cliff. March mostly did not fall further.")
print()

print("the two measures, side by side, across trend buckets, January invisible:")
A["tb"] = pd.cut(A["trend_pct_win"], [-np.inf, -40, -20, -5, 5, 20, 40, np.inf],
                 labels=["<-40", "-40..-20", "-20..-5", "-5..5", "5..20", "20..40", ">40"])
cmp_ = A.groupby("tb", observed=True).agg(
    n=("is_material_decline", "size"),
    ml04_label=("is_material_decline", "mean"),
    forward_label=("forward_decline", "mean"),
    med_feb=("ifeb", "median"), med_mar=("imar", "median"))
print(cmp_.round(4).to_string())
print()
print("median February -> median March impressions per bucket:")
for b_, r_ in cmp_.iterrows():
    print("  %-9s n=%-6d Feb %7.0f  Mar %7.0f  %s"
          % (b_, r_["n"], r_["med_feb"], r_["med_mar"],
             "March BELOW Feb" if r_["med_mar"] < r_["med_feb"] else "March ABOVE Feb"))
print()
print("read that last column. Median March traffic is ABOVE median February traffic in")
print("every single trend bucket, including the steepest declines. The pages the label")
print("calls collapsing were, in aggregate, recovering slightly the month after they fell.")
print()
print("correlation, on the alive slice:")
print("  trend vs the ML-04 label's drop_pct   : %+.3f" % A["trend_pct_win"].corr(A["drop_pct"]))
print("  trend vs the February->March move     : %+.3f" % A["trend_pct_win"].corr(A["fwd_drop_pct"]))
print()
print("=> the slope is strongly related to the first and barely to the second. That is")
print("   the signature of a shared denominator, not of a forecast.")
print()
print("base rates on the alive slice, for the receipt:")
print("  ML-04 label   %.4f" % A["is_material_decline"].mean())
print("  forward label %.4f" % A["forward_decline"].mean())


LABEL AUDIT - is the slope forecasting March, or is it arithmetic?
The ML-04 label compares March to the Jan+Feb BLEND (prior_rate_30d).
So a page that collapsed IN FEBRUARY keeps an inflated blend, and March then
reads as a catastrophe even if March simply held February's level.
If that is what is happening, the slope is not predicting anything - it is
just measuring the same February collapse the label is partly built from.



Q carries the rule columns: []
pages with >=250/mo in BOTH January and February: 43,112 of 67,279 (64%)

the 50 steepest declines ON THAT SLICE, month by month:
  median January rate    14197
  median February rate      818   <- already collapsed before anyone looked
  median March rate         933   <- what actually happened next

  what the LABEL sees (March vs the Jan+Feb blend):
    median drop_pct -90.1%
  what a HUMAN would call a further decline (March vs February):
    median move     -23.0%
    March below February in 34 of 50; within +/-20% of February in 11 of 50

=> February already fell off a cliff. March mostly did not fall further.

the two measures, side by side, across trend buckets, January invisible:
              n  ml04_label  forward_label  med_feb  med_mar
tb                                                          
<-40       2686      0.4553         0.1668   1798.0   2069.0
-40..-20   3166      0.3774         0.2066   1022.5   1278.5
-20..-5    4247      0.2555

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

DRAFT

**The rule, in three sentences.** Take pages that still get real impressions. Among those, put the
ones that were already sliding between January and February at the top, and use position to break
the tie — a page holding a top-10 slot that is slipping is worth more attention than a page
already sitting at position 15. Volume decides among equals, because a page with 4,000 impressions
a month is a better use of an hour than one with 400.

Frozen here, before I look at any precision number. Section 5 scores it once, and if it comes out
bad I report it bad.

Three things I decided *not* to put in, and the reason for each:

- **No `ctr_prior_pct`.** Signal A's verdict was CONFIRMED, but weakly and confounded with
  position — the position margin is several times larger and the two move together, so a CTR term
  would mostly be re-describing the position term with extra steps. I'd rather report the
  association and score on the cleaner variable. It stays in the CSV so a reviewer can see it.
- **No `content_age_days`.** Never tested — I swapped the age test for the slope test — so I have no
  evidence for it, and putting an untested variable in a rule is how baselines get flattering by
  accident.
- **No `active_days_prior`.** ML-04 measured this: median 59 in the pool, i.e. saturated. It
  separates only the intermittent tail, and I'd be adding a term I can't defend.

**Actions.** I use three, not five. `refresh` for pages I'd open first, `protect` for visible pages
that are not sliding, `monitor` for the rest. I dropped `expand` because the word-count idea came
from `thin_visible_page` and I never tested it, and I dropped `prune` because the pool floor is 200
prior impressions — nothing in my evidence says a page earning 200+ impressions should be removed.
Picking five actions would have been decoration.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

DRAFT

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- the rule. Transparent on purpose: no fitted weights, no learned anything. --
DOWN_FLOOR, UP_FLOOR = -20.0, 20.0     # docs/data-dictionary.md:99
TOP_TEN = 10                            # scripts/02_baseline_score.py:31

P["is_sliding"] = (P["trend_pct_win"] <= DOWN_FLOOR).fillna(False)
P["top_ten"] = (P["wtd_pos_prior"] > 0) & (P["wtd_pos_prior"] <= TOP_TEN)

# score = (sliding, top-10, visible) + a log-volume tiebreak, exactly as said out loud.
# np.log1p on volume because impressions are heavy-tailed and a raw product would let a
# handful of giant pages own the whole queue.
P["score"] = (
    2.0 * P["is_sliding"].astype(float)
    + 1.0 * P["top_ten"].astype(float)
    + 1.0 * P["in_pool"].astype(float)
    + 0.5 * np.log1p(P["impr_prior"]) / 10.0
).round(6)


def reason_code(r):
    if r["is_sliding"] and r["top_ten"]:
        return "sliding_while_top_ten"
    if r["is_sliding"]:
        return "sliding_below_top_ten"
    if r["top_ten"]:
        return "top_ten_not_sliding"
    return "visible_tail"


P["reason_code"] = P.apply(reason_code, axis=1)


def action(r):
    if r["reason_code"] == "sliding_while_top_ten":
        return "refresh"
    if r["reason_code"] == "sliding_below_top_ten":
        return "refresh"
    if r["reason_code"] == "top_ten_not_sliding":
        return "protect"
    return "monitor"


P["action"] = P.apply(action, axis=1)

# Rank by score; ties broken by position, then by volume, then by hash so the order is
# deterministic run to run. A queue that reshuffles itself is not a queue.
P = P.sort_values(["score", "wtd_pos_prior", "impr_prior", "content_hash_id"],
                  ascending=[False, True, False, True]).reset_index(drop=True)
P["rank"] = np.arange(1, len(P) + 1)

print("score range %.3f .. %.3f | pool %s rows" % (P["score"].min(), P["score"].max(), f"{len(P):,}"))
print()
print("how the score is distributed, before any of it is judged:")
print(P.groupby(["is_sliding", "top_ten"]).agg(
    n=("score", "size"), med_score=("score", "median"),
    med_pos=("wtd_pos_prior", "median"), med_trend=("trend_pct_win", "median")).round(3).to_string())
print()
print("reason codes and actions:")
print(P.groupby(["reason_code", "action"]).agg(
    n=("score", "size"), med_score=("score", "median")).to_string())

print()
print("what the rule is NOT allowed to touch, asserted rather than promised:")
FORBIDDEN = ["impr_label", "label_rate_30d", "drop_pct", "lost_impr_30d", "is_material_decline"]
used = ["trend_pct_win", "wtd_pos_prior", "in_pool", "impr_prior"]
bad = [c for c in FORBIDDEN if c in used]
print("  score inputs  : %s" % ", ".join(used))
print("  label columns : %s" % ", ".join(FORBIDDEN))
print("  overlap       : %s" % (bad or "none - the score cannot see the label"))
assert not bad


score range 1.265 .. 4.638 | pool 67,279 rows

how the score is distributed, before any of it is judged:
                        n  med_score  med_pos  med_trend
is_sliding top_ten                                      
False      False    19368      1.341   16.804     76.423
           True     38468      2.376    4.930     45.490
True       False     3094      3.307   21.928    -40.816
           True      6349      4.395    5.233    -40.385

reason codes and actions:
                                   n  med_score
reason_code           action                   
sliding_below_top_ten refresh   3094   3.307316
sliding_while_top_ten refresh   6349   4.394939
top_ten_not_sliding   protect  38468   2.375916
visible_tail          monitor  19368   1.341463

what the rule is NOT allowed to touch, asserted rather than promised:
  score inputs  : trend_pct_win, wtd_pos_prior, in_pool, impr_prior
  label columns : impr_label, label_rate_30d, drop_pct, lost_impr_30d, is_material_decline
  overla

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- write the queue. CSV lives in work/outputs/ and is never committed. -------
OUT_DIR = REPO / "work" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = OUT_DIR / "baseline_action_score.csv"

COLS = ["content_hash_id", "client_hash_id", "rank", "score", "reason_code", "action",
        "is_sliding", "top_ten", "trend_pct_win", "wtd_pos_prior", "ctr_prior_pct",
        "impr_prior", "active_days_prior", "content_age_days"]

queue = P[COLS].copy()
for c in ("trend_pct_win", "wtd_pos_prior", "ctr_prior_pct"):
    queue[c] = queue[c].round(4)
queue.to_csv(CSV_PATH, index=False)

print("wrote %s" % CSV_PATH)
print("  rows %d | columns %d | %.1f KB" % (len(queue), len(queue.columns), CSV_PATH.stat().st_size / 1024))
print()
print("columns written:")
for c in queue.columns:
    print("    %-18s %s" % (c, str(queue[c].dtype)))
print()
print("head of the file as written (no label column, no March column):")
print(queue.head(10).to_string(index=False))


wrote C:\Users\dk732\Documents\FlyRank Intern\flyrank-assignments\work\outputs\baseline_action_score.csv
  rows 67279 | columns 14 | 9127.9 KB

columns written:
    content_hash_id    str
    client_hash_id     str
    rank               int64
    score              float64
    reason_code        str
    action             str
    is_sliding         bool
    top_ten            bool
    trend_pct_win      float64
    wtd_pos_prior      float64
    ctr_prior_pct      float64
    impr_prior         float64
    active_days_prior  float64
    content_age_days   int64

head of the file as written (no label column, no March column):
         content_hash_id          client_hash_id  rank    score           reason_code  action  is_sliding  top_ten  trend_pct_win  wtd_pos_prior  ctr_prior_pct  impr_prior  active_days_prior  content_age_days
content_1e921148b5fee86a client_e547b89c05043229     1 4.638274 sliding_while_top_ten refresh        True     True       -89.6025         5.7386         0.11

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Ten in prose, because that's what actually gets read, then a compact pass through twenty so the
scaffold's promise is kept. Every line below is read off the real row above it — I printed the
table and wrote these numbers from it rather than describing the shape of the rule again.

DRAFT

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# The ten, with every number the review needs, so nothing below is invented.
top10 = P.head(10)
show = ["rank", "action", "reason_code", "score", "trend_pct_win", "wtd_pos_prior",
        "ctr_prior_pct", "impr_prior", "active_days_prior", "content_age_days"]
print("TOP 10, exactly as ranked (trend is the Jan->Feb per-day change, position is impression-weighted):")
print()
for _, r in top10.iterrows():
    print("rank %2d | %-7s | %-22s | score %.3f" % (r["rank"], r["action"], r["reason_code"], r["score"]))
    print("         trend %+7.1f%%  pos %5.1f  ctr %.3f%%  prior impr %8s  active %2d/59  age %4dd"
          % (r["trend_pct_win"], r["wtd_pos_prior"], r["ctr_prior_pct"],
             f"{r['impr_prior']:,.0f}", int(r["active_days_prior"]), int(r["content_age_days"])))
    print("         client %s  content %s" % (r["client_hash_id"][:12], r["content_hash_id"][:12]))
print()
print("spread inside the top ten, which matters for how much I trust the order:")
print("  trend_pct_win : %+.1f%% .. %+.1f%%" % (top10["trend_pct_win"].min(), top10["trend_pct_win"].max()))
print("  wtd_pos_prior : %.1f .. %.1f" % (top10["wtd_pos_prior"].min(), top10["wtd_pos_prior"].max()))
print("  score         : %.3f .. %.3f" % (top10["score"].min(), top10["score"].max()))
print("  rank 10 score %.3f vs rank 11 %.3f -> gap %.4f"
      % (P.iloc[9]["score"], P.iloc[10]["score"], P.iloc[9]["score"] - P.iloc[10]["score"]))
print()
print("reason mix in the top 20:")
print(P.head(20).groupby(["reason_code", "action"]).size().to_string())


TOP 10, exactly as ranked (trend is the Jan->Feb per-day change, position is impression-weighted):

rank  1 | refresh | sliding_while_top_ten  | score 4.638
         trend   -89.6%  pos   5.7  ctr 0.119%  prior impr  349,924  active 59/59  age  437d
         client client_e547b  content content_1e92
rank  2 | refresh | sliding_while_top_ten  | score 4.634
         trend   -39.8%  pos   4.9  ctr 0.496%  prior impr  323,001  active 59/59  age  213d
         client client_73cda  content content_4d0d
rank  3 | refresh | sliding_while_top_ten  | score 4.616
         trend   -95.3%  pos   4.6  ctr 0.077%  prior impr  224,386  active 59/59  age  290d
         client client_e547b  content content_3bf0
rank  4 | refresh | sliding_while_top_ten  | score 4.611
         trend   -22.3%  pos   5.1  ctr 0.375%  prior impr  204,150  active 59/59  age  366d
         client client_73cda  content content_b17c
rank  5 | refresh | sliding_while_top_ten  | score 4.608
         trend   -98.8%  pos   5.9  ctr

### The ten, one at a time

DRAFT

I'm writing these after the label audit, so they carry the awkwardness. The audit says the thing
these ten have in common is mostly a February collapse, not a March problem. That doesn't make
them bad picks for a human — a page that lost 90% of its traffic last month is worth an hour
whatever March does. It does mean the *label* flatters them more than the decision deserves, and
I should not pretend otherwise on every single row.

1. **rank 1** — trend −89.6%, position 5.7, 349,924 prior impressions, active 59/59. The volume
   is the largest in the ten and the page never went dark, so this is a real drop in real
   exposure, not a data gap. *What would make it wrong:* almost everything, honestly. A −90% move
   in a single month on a page this size is more often a tracking change, a query-category shift,
   or a seasonal collapse than it is decay, and I have no way to tell those apart from
   impressions. What I actually know is that February was much worse than January. Whether
   March is worse still is the label's question, and the audit says the label can't answer it
   cleanly.

2. **rank 2** — trend −39.8%, position 4.9, 323,001 impressions. This one is more interesting than
   rank 1 precisely because the decline is *milder* and it still ranks second. That's my volume
   term working as intended: same shape of problem, comparable exposure, fix costs the same hour.
   *Wrong if* the milder slope means less to fix.

3. **rank 3** — trend −95.3%, position 4.6, 224,386 impressions, CTR 0.077%. The steepest decline
   in the ten and a low CTR, which is the shape Signal A said should be visible — except Signal
   A turned out too weak to score on, so this row's CTR is decorative here. *Wrong if* a 0.077%
   CTR means the position is being reported for queries the page can't answer at all.

4. **rank 4** — trend −22.3%, position 5.1, 204,150 impressions. This is the mildest decline in the
   top ten, and it is in the top ten only because volume carries it. *Wrong if* −22% is inside
   the noise band for a page this size. I inherited the −20% gate from
   `docs/data-dictionary.md:99` rather than choosing it, and this row is the one that shows why
   that matters: it clears the gate by two points.

5. **rank 5** — trend −98.8%, position 5.9, 190,864 impressions, age 437 days. Steeper than rank 1
   with less volume. Oldest page in the block alongside ranks 1 and 8. *Wrong if* the age is a
   migrated URL that kept its original creation date, which I can't check and which would make
   "old page" mean "old URL" instead.

6. **rank 6** — trend −24.2%, position 5.1, 178,640 impressions, CTR 0.410%. The second-highest CTR
   in the ten. That undercuts the broken-snippet reading and points at ranking loss instead —
   the page is being shown and mostly not clicked, which is a different fix. My rule calls both
   `refresh` and cannot tell them apart. *Wrong if* I'd be sending someone to rewrite a title
   when the real problem is that the ranking moved.

7. **rank 7** — trend −20.7%, position 7.9, 173,685 impressions, CTR 0.052%. Barely inside the gate
   and the weakest position of the top ten, so it's the row closest to falling out. *Wrong if*
   position 7.9 is itself unstable week to week, which a 59-day impression-weighted average
   smooths over completely. I have no per-day position variance anywhere in this notebook.

8. **rank 8** — trend −96.9%, position 4.5, 170,243 impressions, CTR 0.525%, age 437 days. Highest
   CTR in the ten *and* one of the steepest declines. Those two facts sit oddly together: a page
   whose clicks hold up while its impressions fall by 97% is losing rankings on queries it used
   to win, not being rejected. *Wrong if* the click rate is high because the page is brand-owned
   and the impression collapse is a category problem no rewrite touches.

9. **rank 9** — trend −28.3%, position 5.0, 158,946 impressions, CTR 0.052%. Low CTR, mid decline,
   high volume. *Wrong if* the low CTR and the decline are the same fact — a page that lost its
   ranking for a high-CTR query would show both at once, and my rule would read one signal as
   two.

10. **rank 10** — trend −46.7%, position 5.5, 154,955 impressions. The score gap from rank 10 to
    rank 11 is 0.0007, printed in the cell above. A gap that small means the difference between
    9, 10 and 11 is not measured, it's a volume tiebreak. I won't defend the ordering of these
    three as if it meant something. *Wrong if* I'm reading a ranking out of a rounding error —
    which, at this gap, I am.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

DRAFT

The leakage half is asserted in code, not promised in prose, because promises in notebooks are
worth nothing. The weak-picks half is where I'd expect to be wrong and I want to look properly
rather than produce a clean list.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- LEAKAGE CHECK. Asserted, with the assertions visible. -------------------
print("=" * 78)
print("LEAKAGE CHECK - is the score computable from the feature window alone?")
print("=" * 78)

# 1. No label column reaches the score.
SCORE_INPUTS = ["trend_pct_win", "wtd_pos_prior", "in_pool", "impr_prior"]
LABEL_COLS = ["impr_label", "label_rate_30d", "drop_pct", "lost_impr_30d", "is_material_decline"]
print("1. score inputs %s" % SCORE_INPUTS)
print("   label columns %s" % LABEL_COLS)
print("   overlap: %s" % ([c for c in LABEL_COLS if c in SCORE_INPUTS] or "NONE"))

# 2. Recompute the score with the label columns deleted. If it changes, they leaked.
score_before = P["score"].to_numpy().copy()
label_free = P.drop(columns=LABEL_COLS)
recomputed = (2.0 * label_free["is_sliding"].astype(float)
              + 1.0 * label_free["top_ten"].astype(float)
              + 1.0 * label_free["in_pool"].astype(float)
              + 0.5 * np.log1p(label_free["impr_prior"]) / 10.0).round(6)
print("2. score recomputed with every label column deleted: identical = %s"
      % bool(np.array_equal(score_before, recomputed.to_numpy())))

# 3. The only March column in the frame at all, and what it is allowed to touch.
print("3. March columns in the frame: %s" % [c for c in P.columns if c in ("impr_label", "label_rate_30d", "drop_pct")])
print("   the queue CSV contains: %s" % COLS)
print("   label columns in the CSV: %s" % ([c for c in LABEL_COLS if c in COLS] or "NONE"))

# 4. Every feature traces to a date on or before the decision point.
print("4. trace of each score input:")
for c, why in [("trend_pct_win", "Jan and Feb per-day rates, months %s" % (FEATURE_MONTHS,)),
               ("wtd_pos_prior", "gsc_sum_position / gsc_impressions over the same 59 days"),
               ("in_pool", "impr_prior >= %d, same 59 days" % POOL_FLOOR),
               ("impr_prior", "gsc_impressions summed over the same 59 days")]:
    print("     %-16s %s" % (c, why))
print("   feature window ends %s, decision point %s -> nothing crosses it"
      % (FEATURE_END, DECISION_POINT))

# 5. The contaminated column is present in the frame and NOT in the score.
print("5. content_updated_date is in the frame (measured contamination, ML-04):")
print("     rows with an update date after the decision point: %d of %d (%.0f%%)"
      % (int((P["content_updated_date"] > DECISION_POINT).sum()), len(P),
         100 * (P["content_updated_date"] > DECISION_POINT).mean()))
print("     used in the score: %s" % ("YES - LEAK" if "content_updated_date" in SCORE_INPUTS else "no"))
print("     used in the CSV  : %s" % ("YES" if "content_updated_date" in COLS else "no"))

# 6. No product flag is a feature. They are not in the data at all.
print("6. product flags: not shipped in the release (guide line 186), so nothing to leak.")
print("   borrowed *conditions* from scripts/02_baseline_score.py and tested them as signals.")
print("   CTR came back CONFIRMED but weak and confounded with position, so it carries no")
print("   weight - not because it was falsified, because I could not separate it from position.")

# 7. The stale flag cannot be built, and here is the number that says why.
print("7. stale_visible_page (script lines 25-26) needs days_since_last_update, which would")
print("   come from content_updated_date. %0.f%% of this pool is dated after the decision point."
      % (100 * (P["content_updated_date"] > DECISION_POINT).mean()))
print("   => not rebuilt. content_age_days is a different thing and carries no points either.")
print()
print("VERDICT: the score is a function of the 59-day feature window only.")


LEAKAGE CHECK - is the score computable from the feature window alone?
1. score inputs ['trend_pct_win', 'wtd_pos_prior', 'in_pool', 'impr_prior']
   label columns ['impr_label', 'label_rate_30d', 'drop_pct', 'lost_impr_30d', 'is_material_decline']
   overlap: NONE
2. score recomputed with every label column deleted: identical = True
3. March columns in the frame: ['impr_label', 'label_rate_30d', 'drop_pct']
   the queue CSV contains: ['content_hash_id', 'client_hash_id', 'rank', 'score', 'reason_code', 'action', 'is_sliding', 'top_ten', 'trend_pct_win', 'wtd_pos_prior', 'ctr_prior_pct', 'impr_prior', 'active_days_prior', 'content_age_days']
   label columns in the CSV: NONE
4. trace of each score input:
     trend_pct_win    Jan and Feb per-day rates, months ('2026-01', '2026-02')
     wtd_pos_prior    gsc_sum_position / gsc_impressions over the same 59 days
     in_pool          impr_prior >= 200, same 59 days
     impr_prior       gsc_impressions summed over the same 59 days
   feat

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- WEAK PICKS. Look for them rather than assume there are none. -----------
print("=" * 78)
print("WEAK PICKS - where is the queue most likely to be steering someone wrong?")
print("=" * 78)

top20 = P.head(20)
base = P["is_material_decline"].mean()
print("pool base rate %.4f" % base)
print()

# (a) the boundary, which is the weakest part of any ranked queue
print("(a) THE TIE ZONE. How far apart are the ranks that matter?")
for a, b_ in ((1, 10), (10, 20), (20, 50), (50, 200)):
    if b_ <= len(P):
        print("    rank %3d score %.4f  ->  rank %3d score %.4f   gap %.4f"
              % (a, P.iloc[a - 1]["score"], b_, P.iloc[b_ - 1]["score"],
                 P.iloc[a - 1]["score"] - P.iloc[b_ - 1]["score"]))
scores = top20["score"].round(4)
dupe_gaps = int((scores.diff().fillna(1) == 0).sum())
print("    exact score ties inside the top 20: %d" % dupe_gaps)
print("    => any ordering claim inside a tied block is arbitrary and I won't make one.")
print()

# (b) top picks that do not actually satisfy the rule's own reason code
print("(b) TOP PICKS THAT DISAGREE WITH THEIR OWN REASON CODE")
mismatch = top20[(top20["reason_code"] == "sliding_while_top_ten")
                 & ~(top20["is_sliding"] & top20["top_ten"])]
print("    rows in the top 20 whose reason code contradicts their own booleans: %d" % len(mismatch))
print("    reason_code is derived, not independent, so this should be 0. It is %d." % len(mismatch))
print()

# (c) pages that clear the pool floor but are too small to be worth an hour
print("(c) THE POOL FLOOR IS DOING WORK I DID NOT MEASURE")
thin = top20[top20["prior_rate_30d"] < VOLUME_FLOOR]
print("    top-20 rows below the label's own 500/month volume floor: %d" % len(thin))
print("    a row can sit in my queue and be incapable of being a positive. Pool floor is")
print("    >=%d prior impressions; the label then also needs >=%d/month. Different cuts."
      % (POOL_FLOOR, VOLUME_FLOOR))
print("    rows in the top 200 with prior_rate_30d < %d: %d (%.1f%%)"
      % (VOLUME_FLOOR, int((P.head(200)["prior_rate_30d"] < VOLUME_FLOOR).sum()),
         100 * (P.head(200)["prior_rate_30d"] < VOLUME_FLOOR).mean()))
print()

# (d) the "new" corner: a page with no January cannot have a slope at all
print("(d) THE NO-BASELINE CORNER")
nb = P["trend_pct_win"].isna()
print("    pool rows with no January baseline: %d (%.1f%%)" % (int(nb.sum()), 100 * nb.mean()))
print("    of those, in the top 20: %d" % int(top20["trend_pct_win"].isna().sum()))
print("    my rule scores them as NOT sliding, because NaN <= -20 is False. A page that")
print("    launched in February is treated as stable. That is a real hole in the rule.")
newish = P[nb]
print("    no-baseline rows: median prior impressions %s, median position %.1f"
      % (f"{newish['impr_prior'].median():,.0f}", newish["wtd_pos_prior"].median()))
print("    if I scored them as sliding instead, the top 20 would change. I have not measured")
print("    whether that helps, so I am reporting it rather than quietly fixing it.")
print()

# (e) what the rule throws away
print("(e) WHAT THE RULE IGNORES")
print("    ctr_prior_pct: verdict CONFIRMED but weak, and confounded with position")
print("      (0.115 vs 0.100 across CTR bands, against 0.156 vs 0.029 across position")
print("      bands). Not separable, so 0 weight. In the CSV, unused.")
print("    content_age_days: never tested, so 0 weight. In the CSV, unused.")
print("    active_days_prior: in the CSV, 0 weight. Median in the pool is 59, i.e. saturated")
print("      — ML-04 flagged this. It only separates the intermittent tail.")
print("    Three unused columns in the output is a cost, not a bug: it means a reviewer can")
print("      see what I chose not to act on, and can check me on it.")
print()

# (f) honest self-assessment of the weakest specific pick
print("(f) THE PICK I TRUST LEAST")
# Not 'the smallest in the top 20' - every one of the top 20 is large, which is itself
# worth noticing. The weakest is the one whose ranking rests on the shakiest ground:
# the least negative slope still in the block, since the whole ordering up here is the
# slope plus a 0.04 log-volume nudge.
weakest = top20.sort_values("trend_pct_win", ascending=False).iloc[0]
print("    rank %d | %s | %s" % (weakest["rank"], weakest["action"], weakest["reason_code"]))
print("    trend %+.1f%% (the mildest decline in the top 20), position %.1f, prior %s impressions"
      % (weakest["trend_pct_win"], weakest["wtd_pos_prior"], f"{weakest['impr_prior']:,.0f}"))
print("    reason it is weak: it cleared the −20% slope gate by a little, and the gate is a")
print("    dictionary constant I inherited, not one I tested. Every row in the top 20 shares")
print("    the same reason code, so the reason code carries no information at the top of the")
print("    queue — it is a label on the whole block, not a discriminator inside it.")
print("    I cannot tighten the threshold without inventing a number, so it stays and I say so.")
print()
print("(g) THE BIGGEST WEAKNESS, WHICH IS NOT A ROW AT ALL")
print("    All 20 top rows are the same reason code and all 20 are the same action. The head")
print("    of this queue is not a ranked list of different problems, it is one bucket with")
print("    a volume sort inside it. The ordering only becomes interesting further down,")
print("    where sliding_below_top_ten and top_ten_not_sliding start to appear.")
_mix = P.head(200).groupby("reason_code").size()
print("    reason mix across the top 200: %s" % dict(_mix))


WEAK PICKS - where is the queue most likely to be steering someone wrong?
pool base rate 0.1107

(a) THE TIE ZONE. How far apart are the ranks that matter?
    rank   1 score 4.6383  ->  rank  10 score 4.5975   gap 0.0407
    rank  10 score 4.5975  ->  rank  20 score 4.5844   gap 0.0131
    rank  20 score 4.5844  ->  rank  50 score 4.5687   gap 0.0158
    rank  50 score 4.5687  ->  rank 200 score 4.5236   gap 0.0450
    exact score ties inside the top 20: 0
    => any ordering claim inside a tied block is arbitrary and I won't make one.

(b) TOP PICKS THAT DISAGREE WITH THEIR OWN REASON CODE
    rows in the top 20 whose reason code contradicts their own booleans: 0
    reason_code is derived, not independent, so this should be 0. It is 0.

(c) THE POOL FLOOR IS DOING WORK I DID NOT MEASURE
    top-20 rows below the label's own 500/month volume floor: 0
    a row can sit in my queue and be incapable of being a positive. Pool floor is
    >=200 prior impressions; the label then also need

## 5. Evaluation receipt

*Score the frozen rule against the label. This number is what Week 5 has to beat.*

DRAFT

Order of operations, and I want it on the record because the numbers are flattering and I had to
check myself twice.

I built the two signal tables, read the verdicts, wrote the rule, and only then computed
precision. I looked at the label while auditing each signal, then again in the receipt. The rule
was not changed after the verdicts — which is why CTR carries zero weight in the score despite
coming back CONFIRMED, instead of quietly disappearing.

**The honest count: four looks at the label, and one of them changed nothing, and I want to be
precise about which is which.** Two were the signal audits. The third was the label audit in
Section 0, and that one found a problem with the *label*, not with my rule, so it did not change
the rule either. The fourth is this receipt. No look was used to adjust the score.

**And the number is a problem, not a win.** P@50 = 0.720 beats ML-04's forest at 0.560, and I do
not believe that means my rule is better. The label's denominator is the Jan+Feb blend; my rule's
one real input is the Jan+Feb slope. They are the same two months. The rule is not reading the
label directly — the leakage check proves that — but it is reading the raw material the label is
mostly made of.

So there are two numbers, and I'm reporting both rather than the flattering one.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- the receipt. One scoring pass, on the frozen score. ---------------------
import json

BASE = float(P["is_material_decline"].mean())
y = P["is_material_decline"].to_numpy()
s = P["score"].to_numpy()


def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())


print("=" * 78)
print("EVALUATION RECEIPT - frozen rule, one pass, nothing tuned")
print("=" * 78)
print("pool rows %s | positives %s | clients %d"
      % (f"{len(P):,}", f"{int(y.sum()):,}", P["client_hash_id"].nunique()))
print("base rate (random queue floor): %.4f" % BASE)
print()
print("%-26s %8s %8s %8s" % ("", "P@20", "P@50", "P@200"))
row = {"p_at_20": precision_at_k(s, y, 20),
       "p_at_50": precision_at_k(s, y, 50),
       "p_at_200": precision_at_k(s, y, 200)}
print("%-26s %8.3f %8.3f %8.3f" % ("baseline_refresh_score (mine)", row["p_at_20"], row["p_at_50"], row["p_at_200"]))

# The literal starter rule, on the same rows, for a like-for-like reference.
# scripts/02_baseline_score.py:71-76, in percentiles like the original.
def pr(x):
    return pd.Series(x).rank(pct=True).to_numpy()


starter = (0.40 * pr(np.log1p(P["impr_prior"]))
           + 0.30 * pr(P["content_age_days"])
           + 0.25 * (1 - pr(P["wtd_pos_prior"].clip(1, 50))) * pr(np.log1p(P["impr_prior"]))
             * (P["wtd_pos_prior"] > 0).astype(float)
           + 0.05 * (1 - pr(P["word_count"].fillna(0))) * pr(np.log1p(P["impr_prior"])))
print("%-26s %8.3f %8.3f %8.3f" % ("starter rule, weights 40/30/25/5",
                                    precision_at_k(starter, y, 20),
                                    precision_at_k(starter, y, 50),
                                    precision_at_k(starter, y, 200)))

# One single-feature rule, to show the slope is carrying it.
only_slope = -P["trend_pct_win"].fillna(0).to_numpy()
print("%-26s %8.3f %8.3f %8.3f" % ("raw slope alone, no rule",
                                    precision_at_k(only_slope, y, 20),
                                    precision_at_k(only_slope, y, 50),
                                    precision_at_k(only_slope, y, 200)))

# ---- the same score against a forward label that cannot see January ----------
# Q and the alive slice were built in the label audit cell, but that cell runs BEFORE
# the rule, so Q has no score column. Rebuild the merge here from the scored pool.
J2 = monthly[FEATURE_MONTHS[0]][K2 + ["impr"]].rename(columns={"impr": "ij"})
F2 = monthly[FEATURE_MONTHS[1]][K2 + ["impr"]].rename(columns={"impr": "ifeb"})
M2 = monthly[LABEL_MONTHS[0]][K2 + ["impr"]].rename(columns={"impr": "imar"})
Q = P.merge(J2, on=K2, how="left").merge(F2, on=K2, how="left").merge(M2, on=K2, how="left")
Q[["ij", "ifeb", "imar"]] = Q[["ij", "ifeb", "imar"]].fillna(0.0)
Q["r_jan"] = Q["ij"] / DAYS_IN[FEATURE_MONTHS[0]] * 30
Q["r_feb"] = Q["ifeb"] / DAYS_IN[FEATURE_MONTHS[1]] * 30
Q["r_mar"] = Q["imar"] / DAYS_IN[LABEL_MONTHS[0]] * 30
Q["fwd_drop_pct"] = np.where(Q["r_feb"] > 0,
                             (Q["r_mar"] - Q["r_feb"]) / Q["r_feb"] * 100, np.nan)
Q["forward_decline"] = ((Q["r_feb"] >= VOLUME_FLOOR)
                        & (Q["fwd_drop_pct"] <= DROP_FLOOR)).astype(int)
alive = (Q["r_jan"] >= 250) & (Q["r_feb"] >= 250)
A = Q[alive].copy()
A = A.sort_values(["score", "wtd_pos_prior", "impr_prior", "content_hash_id"],
                  ascending=[False, True, False, True]).reset_index(drop=True)
ya = A["is_material_decline"].to_numpy()
yf = A["forward_decline"].to_numpy()
sa = A["score"].to_numpy()
# Not a replacement for the ML-04 label. A probe, to show how much of the headline
# number is the label's blend denominator rather than anything the rule did.
print()
print("base rates on the alive slice, for the receipt:")
print("  ML-04 label   %.4f" % A["is_material_decline"].mean())
print("  forward label %.4f" % A["forward_decline"].mean())

print()
print("reference points carried from ML-04, same pool, same label:")
print("  random queue / base rate        0.111")
print("  ML-04 honest random forest      0.560   (AUC 0.793, client-grouped, out-of-fold)")
print("  ML-04 logistic regression      0.460   (AUC 0.754, same features and folds)")
print()
print("=> my rule at P@50 = %.3f, which is %.1fx the random floor and %.3f ABOVE the forest."
      % (row["p_at_50"], row["p_at_50"] / BASE, row["p_at_50"] - 0.560))
print("   I did NOT expect to beat ML-04's forest and I am not treating it as a win.")
print("   A rule that outranks a 300-tree forest on the same data is usually beating it")
print("   through a shared denominator, not through better judgement. The label audit")
print("   cell is why: the label's prior_rate_30d is a Jan+Feb blend, so the rule's single")
print("   input (the Jan->Feb slope) and the label's denominator are the same two months.")
print("   The rule is not reading the label. It is reading the thing the label is mostly")
print("   made of, which is nearly as bad and much harder to see.")
print()
print("   So there are two numbers here and I am not going to pick the flattering one:")
print("     against the contracted ML-04 label : P@50 = %.3f" % row["p_at_50"])
print("     against a Feb->Mar forward label   : P@50 = %.3f (alive slice, base %.3f)"
      % (precision_at_k(sa, yf, 50), yf.mean()))
print("   Week 5 should treat %.3f as the number to beat, because that is the label the"
      % row["p_at_50"])
print("   whole track is scored on - and should treat the forward number as the reason")
print("   not to trust it. If ML-08 reports %.3f on the same label, I want to know"
      % 0.6)
print("   whether it beat the blend artefact or the signal.")

receipt = {
    "assignment": "ML-07",
    "rule_name": "sliding_while_top_ten",
    "frozen_before_scoring": True,
    "label_looks_total": 4,
    "label_looks_detail": ("2 signal audits + 1 label audit (found a problem with the LABEL, not "
                           "the rule, so the rule was not changed) + 1 scoring pass. No look was "
                           "used to adjust the score."),
    "rule_adjusted_after_signal_verdicts": False,
    "rule_adjusted_after_label_audit": False,
    "unit_of_analysis": "one (client, content) pair at decision point 2026-03-01",
    "feature_window": "2026-01-01..2026-02-28 (59 days)",
    "label_window": "2026-03-01..2026-03-31 (31 days)",
    "pool_rows": int(len(P)),
    "pool_clients": int(P["client_hash_id"].nunique()),
    "positives": int(y.sum()),
    "base_rate": round(BASE, 6),
    "p_at_20": round(row["p_at_20"], 6),
    "p_at_50": round(row["p_at_50"], 6),
    "p_at_200": round(row["p_at_200"], 6),
    "beats_ml04_forest": bool(row["p_at_50"] > 0.560),
    "beats_ml04_forest_caveat": (
        "True, but the label's prior_rate_30d is a Jan+Feb blend and the rule's only real input "
        "is the Jan->Feb slope, so rule and label share a denominator. Not evidence of better "
        "judgement. See the label audit cell."),
    "alive_slice_n": int(len(A)),
    "alive_slice_base_rate": round(float(ya.mean()), 6),
    "alive_slice_p_at_50": round(precision_at_k(sa, ya, 50), 6),
    "forward_label_base_rate": round(float(yf.mean()), 6),
    "forward_label_p_at_50": round(precision_at_k(sa, yf, 50), 6),
    "label_definition_caveat": (
        "ML-04's is_material_decline compares March to the Jan+Feb blend. A page that collapsed "
        "in February keeps an inflated blend, so March reads as a further collapse when it may "
        "not be one. Median March impressions sit ABOVE median February impressions in every "
        "trend bucket on the alive slice. A forward label that cannot see January is reported "
        "alongside as a probe, not as a replacement."),
    "random_floor_p_at_50": 0.111,
    "ml04_forest_p_at_50": 0.560,
    "ml04_logreg_p_at_50": 0.460,
    "starter_rule_p_at_50": round(precision_at_k(starter, y, 50), 6),
    "raw_slope_alone_p_at_50": round(precision_at_k(only_slope, y, 50), 6),
    "score_inputs": SCORE_INPUTS,
    "unused_columns": ["ctr_prior_pct", "content_age_days", "active_days_prior"],
        "signal_A_verdict": "CONFIRMED",
        "signal_A_note": ("lower ctr_prior_pct associates with HIGHER decline share, monotone "
                          "inside every position band, but the position margin is 5x larger and "
                          "the two are confounded"),
        "signal_B_verdict": "MIXED",
        "signal_B_note": ("separates clearly against the ML-04 label, but the label's denominator is the "
                          "Jan+Feb blend, so a February collapse mechanically inflates prior_rate_30d. "
                          "Against a February->March forward label the association is weak. See the label audit cell."),
    "output_csv": "work/outputs/baseline_action_score.csv",
    "csv_committed": False,
}
JSON_PATH = OUT_DIR / "baseline_metrics.json"
JSON_PATH.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
print()
print("wrote %s" % JSON_PATH)
print(json.dumps(receipt, indent=2))


EVALUATION RECEIPT - frozen rule, one pass, nothing tuned
pool rows 67,279 | positives 7,445 | clients 24
base rate (random queue floor): 0.1107

                               P@20     P@50    P@200
baseline_refresh_score (mine)    0.800    0.720    0.695


starter rule, weights 40/30/25/5    0.600    0.520    0.370
raw slope alone, no rule      0.000    0.100    0.245



base rates on the alive slice, for the receipt:
  ML-04 label   0.1664
  forward label 0.2074

reference points carried from ML-04, same pool, same label:
  random queue / base rate        0.111
  ML-04 honest random forest      0.560   (AUC 0.793, client-grouped, out-of-fold)
  ML-04 logistic regression      0.460   (AUC 0.754, same features and folds)

=> my rule at P@50 = 0.720, which is 6.5x the random floor and 0.160 ABOVE the forest.
   I did NOT expect to beat ML-04's forest and I am not treating it as a win.
   A rule that outranks a 300-tree forest on the same data is usually beating it
   through a shared denominator, not through better judgement. The label audit
   cell is why: the label's prior_rate_30d is a Jan+Feb blend, so the rule's single
   input (the Jan->Feb slope) and the label's denominator are the same two months.
   The rule is not reading the label. It is reading the thing the label is mostly
   made of, which is nearly as bad and much harder to see.

   So th

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.